In [1]:
%%capture
!pip install pyspark
!pip install findspark
!pip install pyngrok

In [2]:
import getpass
import findspark
import pandas as pd
from pyspark.sql import types
from pyngrok import ngrok, conf
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, LongType, StringType

findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName('testColab') \
        .getOrCreate()

In [3]:
print("Enter your authtoken, which can be copied "
"from https://dashboard.ngrok.com/get-started/your-authtoken")
conf.get_default().auth_token = getpass.getpass()

ui_port = 4040
public_url = ngrok.connect(ui_port).public_url
print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{ui_port}\"")

Enter your authtoken, which can be copied from https://dashboard.ngrok.com/get-started/your-authtoken
··········


 * ngrok tunnel "https://postelementary-unactivated-zola.ngrok-free.dev" -> "http://127.0.0.1:4040"


In [4]:
#Check if AQE is enabled

spark.conf.get("spark.sql.adaptive.enabled")

'true'

In [5]:
spark.conf.set("spark.sql.adaptive.enabled", "false")

In [6]:
# Confirm AQE is disabled

spark.conf.get("spark.sql.adaptive.enabled")

'false'

In [7]:
df = spark.read.option("inferSchema", "true") \
          .option("header", "true") \
          .csv("/content/BigMart Sales.csv")

df.show(2)

+---------------+-----------+----------------+---------------+-----------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|  Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+-----------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|      Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          DRC01|       5.92|         Regular|    0.019278216|Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Supermarket Type2|         443.4228|
+--------------

In [8]:
# How many partitions does our file have now?

df.rdd.getNumPartitions()

1

In [9]:
# Default partition size is 120mb - my data is less hence why it has 1 partition, reduce the defaulf partition size to 120kb

spark.conf.set("spark.sql.files.maxPartitionBytes", 131072)

In [10]:
# Read data again and check partitions

df = spark.read.option("inferSchema", "true") \
          .option("header", "true") \
          .csv("/content/BigMart Sales.csv")

df.rdd.getNumPartitions()

7

In [11]:
# You can also use repartitions to increase the partitions
df = df.repartition(10)

In [12]:
# View what partitions each role of data falls into

df.withColumn("partition_id", spark_partition_id()).show(2)

+---------------+-----------+----------------+---------------+------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|   Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|partition_id|
+---------------+-----------+----------------+---------------+------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+------------+
|          FDS52|       NULL|         Low Fat|    0.005448005|Frozen Foods|102.1016|           OUT027|                     1985|     Medium|              Tier 3|Supermarket Type3|         3542.056|           0|
|          FDL22|      16.85|         Low Fat|    0.036390174| Snack Foods| 91.4488|           OUT046|                     1997|      Small|              Ti

In [13]:
# Check how many files are read without scanning optimisation

df.write.mode("overwrite").parquet("df_not_optimised")

In [14]:
df_not_optimised = spark.read.format("parquet").load("/content/df_not_optimised")

df_not_optimised = df_not_optimised.filter(col("Outlet_Location_Type")=='Tier 1')

In [15]:
df_not_optimised.show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDK14|       6.98|         Low Fat|    0.041105789|              Canned| 83.4934|           OUT046|                     1997|      Small|              Tier 1|Supermarket Type1|        1555.9746|
|          FDJ08|       NULL|         Low Fat|    0.193772568|Fruits and Vegeta...|190.3846|           OUT019|                     1985|      Small|              Tier 1|    Gro

# Apply Scanning optimisation

In [16]:
# Check how many files are read with scanning optimisation

df.write.mode("overwrite").partitionBy("Outlet_Location_Type").parquet("df_optimised")

df_optimised = spark.read.format("parquet").load("/content/df_optimised")

df_optimised = df_optimised.filter(col("Outlet_Location_Type")=='Tier 1')

In [17]:
df_optimised.show()

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+-----------------+-----------------+--------------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|      Outlet_Type|Item_Outlet_Sales|Outlet_Location_Type|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+-----------------+-----------------+--------------------+
|          FDH28|      15.85|         Regular|    0.110030997|        Frozen Foods| 37.2506|           OUT046|                     1997|      Small|Supermarket Type1|         265.6542|              Tier 1|
|          NCW53|       NULL|         Low Fat|    0.053392944|  Health and Hygiene|193.8162|           OUT019|                     1985|      Small|    Grocery Store|         3

# Joins Optimisation

#

In [18]:
df_transactions = spark.createDataFrame([
    (1, "US", 100),
    (2, "UK", 200),
    (3, "IN", 300),
    (4, "UK", 400),
    (5, "US", 500),
    (6, "UK", 600),
],  ["id", "country_code", "amount"])



df_countries = spark.createDataFrame([
    ("US", "United States"),
    ("IN", "India"),
    ("UK", "United Kingdom"),
 ], ["country_code", "country_name"])


In [19]:
df_join = df_transactions.join(df_countries, df_transactions['country_code']==df_countries['country_code'], how='inner')

In [20]:
df_join.show()

+---+------------+------+------------+--------------+
| id|country_code|amount|country_code|  country_name|
+---+------------+------+------------+--------------+
|  1|          US|   100|          US| United States|
|  5|          US|   500|          US| United States|
|  3|          IN|   300|          IN|         India|
|  2|          UK|   200|          UK|United Kingdom|
|  4|          UK|   400|          UK|United Kingdom|
|  6|          UK|   600|          UK|United Kingdom|
+---+------------+------+------------+--------------+



In [21]:
# Optimise the join - make sure if you want to use broadcast, you wrap it around the small table. Instead of sort merge joins, Spar performed Broadcast Exchange

df_join_optimised = df_transactions.join(broadcast(df_countries), df_transactions['country_code']==df_countries['country_code'], how='inner')
df_join_optimised.show()

+---+------------+------+------------+--------------+
| id|country_code|amount|country_code|  country_name|
+---+------------+------+------------+--------------+
|  1|          US|   100|          US| United States|
|  2|          UK|   200|          UK|United Kingdom|
|  3|          IN|   300|          IN|         India|
|  4|          UK|   400|          UK|United Kingdom|
|  5|          US|   500|          US| United States|
|  6|          UK|   600|          UK|United Kingdom|
+---+------------+------+------------+--------------+



#SQL Hints

In [22]:
df_transactions.createOrReplaceTempView('transactions')
df_countries.createOrReplaceTempView('countries')

In [23]:
# Join sql tables without optimisation - sortmerge

df_sql_joined = spark.sql("""
                select * from transactions t
                join countries c
                on t.country_code = c.country_code
                """)
df_sql_joined.show()

+---+------------+------+------------+--------------+
| id|country_code|amount|country_code|  country_name|
+---+------------+------+------------+--------------+
|  1|          US|   100|          US| United States|
|  5|          US|   500|          US| United States|
|  3|          IN|   300|          IN|         India|
|  2|          UK|   200|          UK|United Kingdom|
|  4|          UK|   400|          UK|United Kingdom|
|  6|          UK|   600|          UK|United Kingdom|
+---+------------+------+------------+--------------+



In [24]:
# Join sql tables with optimisation - broadcast Exchange

df_sql_joined_opt = spark.sql("""
                select * /* broadcast(c) */
                from transactions t
                join countries c
                on t.country_code = c.country_code
                """)
df_sql_joined_opt.show()

+---+------------+------+------------+--------------+
| id|country_code|amount|country_code|  country_name|
+---+------------+------+------------+--------------+
|  1|          US|   100|          US| United States|
|  5|          US|   500|          US| United States|
|  3|          IN|   300|          IN|         India|
|  2|          UK|   200|          UK|United Kingdom|
|  4|          UK|   400|          UK|United Kingdom|
|  6|          UK|   600|          UK|United Kingdom|
+---+------------+------+------------+--------------+



#Caching & Persistence

In [25]:

# For everytime df is called, it recalls the read of df and recalculates df

df_sorted = df.filter(col("Outlet_Location_Type")=="Tier 1")

df_sorted2 = df.filter(col("Outlet_Location_Type")=="Tier 2")

In [26]:
df_sorted2.show(1)

+---------------+-----------+----------------+---------------+------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|   Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDE24|      14.85|         Low Fat|     0.09344495|Baking Goods|141.0812|           OUT035|                     2004|      Small|              Tier 2|Supermarket Type1|        1139.8496|
+---------------+-----------+----------------+---------------+------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
only showi

In [27]:
# To cache tables, simply cache during table reads

df_f = spark.read.option("inferSchema", "true") \
          .option("header", "true") \
          .csv("/content/BigMart Sales.csv") \
          .cache()


df_f.show(1)


+---------------+-----------+----------------+---------------+---------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+---------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|    Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
+---------------+-----------+----------------+---------------+---------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
only showing top 1 row


In [28]:
# To uncache
df_f.unpersist()


DataFrame[Item_Identifier: string, Item_Weight: double, Item_Fat_Content: string, Item_Visibility: double, Item_Type: string, Item_MRP: double, Outlet_Identifier: string, Outlet_Establishment_Year: int, Outlet_Size: string, Outlet_Location_Type: string, Outlet_Type: string, Item_Outlet_Sales: double]

In [29]:
# To use specific persist methods

from pyspark.storagelevel import StorageLevel

df1 = df.persist(StorageLevel.MEMORY_ONLY)
df1.unpersist()

DataFrame[Item_Identifier: string, Item_Weight: double, Item_Fat_Content: string, Item_Visibility: double, Item_Type: string, Item_MRP: double, Outlet_Identifier: string, Outlet_Establishment_Year: int, Outlet_Size: string, Outlet_Location_Type: string, Outlet_Type: string, Item_Outlet_Sales: double]

#Dynamic Resource Allocation

In [30]:
# spark-submit \
#  --conf spark.shuffle.service.enabled=true \
#  --conf spark.dynamicAllocation.enabled=true \
#  --conf spark.dynamicAllocation.minExecutors=0 \
#  --conf spark.dynamicAllocation.initialExecutors=1 \
#  --conf spark.dynamicAllocation.maxExecutors=20

#Adaptive Query Optimisation - AQE

In [31]:
# Apply groupby function of df - partitions turned to 200 due to wide transformation and AQE is off

item_fat = df.groupBy("Item_Fat_Content").count()

item_fat.show()

+----------------+-----+
|Item_Fat_Content|count|
+----------------+-----+
|         low fat|  112|
|         Low Fat| 5089|
|              LF|  316|
|         Regular| 2889|
|             reg|  117|
+----------------+-----+



In [32]:
# Turn AQE on

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.get("spark.sql.adaptive.enabled")

'true'

In [33]:
# Apply groupby function of df - with AQE turned on (here the number of partitions is 1)

item_fat = df.groupBy("Item_Fat_Content").count()

item_fat.show()

+----------------+-----+
|Item_Fat_Content|count|
+----------------+-----+
|         low fat|  112|
|         Low Fat| 5089|
|              LF|  316|
|         Regular| 2889|
|             reg|  117|
+----------------+-----+



#Dynamic Partition Pruning

In [45]:
# Turn off AQE, DPP, and AutoBroadcast

spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [59]:
# Read data again and check partitions

df_new = spark.read.option("inferSchema", "true") \
          .option("header", "true") \
          .csv("/content/BigMart Sales.csv")

In [60]:
# Create partioned data

df_partition = df_new.limit(10)


df_partition.write.mode("overwrite").partitionBy("Item_Identifier").parquet("df_new_partitioned")

In [61]:
# Create none partitioned data

df_partition.write.mode("overwrite").parquet("df_new_no_partition")

In [62]:
# Load partioned and none partitioned data


df_part = spark.read.option("inferSchema", "true").option("header", "true").parquet("/content/df_new_partitioned")

df_no_part = spark.read.option("inferSchema", "true").option("header", "true").parquet("/content/df_new_no_partition")

In [63]:
df_join = df_part.join(df_no_part.filter(col("Outlet_Type")== 'Grocery Store'), df_part['Item_Identifier']==df_no_part['Item_Identifier'], how="inner")
df_join.show(2)

+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-------------+-----------------+---------------+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-------------+-----------------+
|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|  Outlet_Type|Item_Outlet_Sales|Item_Identifier|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|  Outlet_Type|Item_Outlet_Sales|
+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-------------+-----------------+-----------

In [64]:
# Turn on Dynamic Partition Pruning

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [65]:
# Load partioned and none partitioned data


df_join = df_part.join(df_no_part.filter(col("Item_Identifier")== 'FDX07'), df_part['Item_Identifier']==df_no_part['Item_Identifier'], how="inner")
df_join.show(2)

+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-------------+-----------------+---------------+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-------------+-----------------+
|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|  Outlet_Type|Item_Outlet_Sales|Item_Identifier|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|  Outlet_Type|Item_Outlet_Sales|
+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-------------+-----------------+-----------

# Broacast Variable

In [69]:
# Create new dataframe

df = spark.createDataFrame([
    ('1001',),
    ('1002',),
    ('1003',)
], ["product_id"])


# create lookup dict
product_dict = {
    "1001": "iphone",
    "1002": "samsung",
    "1003": "pixel"
}

In [70]:
# Broadcast dictionary variable

broad_var = spark.sparkContext.broadcast(product_dict)

In [71]:
broad_var.value

{'1001': 'iphone', '1002': 'samsung', '1003': 'pixel'}

In [72]:
broad_var.value.get("1001")

'iphone'

In [74]:
# Define UDF

def my_map(x):
  return broad_var.value.get(x)